# Hong Kong Daily Temperature — Levels 1 and 2
Abhinav Sheshadri

## 1. Setup and visualization settings

In [2]:
from pathlib import Path
import calendar

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Keep the CSV beside this notebook
CSV_PATH = Path("temperature_daily.csv")
LEVEL2_YEAR_COUNT = 10

# Set the chart sizes and spacing to match the examples
LEVEL1_SIZE = (1010, 510)
LEVEL2_SIZE = (1146, 945)
LEVEL1_GAP = (8, 8)
LEVEL2_GAP = (18, 14)

# The color palette matches the reference legend
TEMPERATURE_COLORS = [
    "#5e4fa2", "#3288bd", "#66c2a5", "#abdda4", "#e6f598",
    "#ffffbf", "#fee08b", "#fdae61", "#f46d43", "#d53e4f", "#9e0142",
]
COLOR_BIN_EDGES = [0, 4, 7, 10, 13, 17, 22, 25, 29, 32, 36, 40]
TEMPERATURE_RANGE = (0, 40)
DAILY_MAX_COLOR = "#31a354"
DAILY_MIN_COLOR = "#9ecae1"
MONTHS = list(range(1, 13))
MONTH_NAMES = [calendar.month_name[m] for m in MONTHS]

# Check the settings and make sure the file exists
assert 5 <= LEVEL2_YEAR_COUNT <= 10
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        "Place temperature_daily.csv beside this notebook, then run it again."
    )

## 2. Load and validate the CSV

In [3]:
def load_temperatures(path: Path) -> pd.DataFrame:
    """Load valid daily records and derive year/month/day calendar columns."""
    daily = pd.read_csv(path)
    # Make sure the columns we need are there
    required = {"date", "max_temperature", "min_temperature"}
    missing = required.difference(daily.columns)
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")

    # Convert the dates and temperatures to the right types
    daily = daily[["date", "max_temperature", "min_temperature"]].copy()
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    for metric in ("max_temperature", "min_temperature"):
        daily[metric] = pd.to_numeric(daily[metric], errors="coerce")

    # Reject missing values duplicate dates and invalid temperatures
    if daily.isna().any().any():
        raise ValueError("CSV contains missing or invalid dates/temperatures")
    if daily["date"].duplicated().any():
        raise ValueError("CSV contains duplicate dates")
    if (daily["max_temperature"] < daily["min_temperature"]).any():
        raise ValueError("A daily maximum is less than the daily minimum")

    # Sort the dates before splitting them into calendar fields
    daily = daily.sort_values("date").reset_index(drop=True)
    daily["year"] = daily["date"].dt.year
    daily["month"] = daily["date"].dt.month
    daily["day"] = daily["date"].dt.day
    return daily


daily_temperatures = load_temperatures(CSV_PATH)
print(f"Loaded {len(daily_temperatures):,} days from "
      f"{daily_temperatures['date'].min().date()} through "
      f"{daily_temperatures['date'].max().date()}.")
display(daily_temperatures.head())

Loaded 7,532 days from 1997-01-01 through 2017-10-28.


,date,max_temperature,min_temperature,year,month,day
0,1997-01-01,25,16,1997,1,1
1,1997-01-02,22,18,1997,1,2
2,1997-01-03,18,17,1997,1,3
3,1997-01-04,20,17,1997,1,4
4,1997-01-05,19,15,1997,1,5


## 3. Calculate monthly statistics

**Monthly maximum** is the highest *daily* maximum within the month. **Monthly minimum** is the lowest *daily* minimum. The complete grid preserves cells with no observations so they can be drawn as blanks.

In [4]:
def summarize_months(daily: pd.DataFrame) -> pd.DataFrame:
    """Aggregate extrema and observed-day counts into a complete year/month grid."""
    # Find the highest and lowest readings in each month
    observed = (daily.groupby(["year", "month"])
        .agg(monthly_max=("max_temperature", "max"),
             monthly_min=("min_temperature", "min"),
             observed_days=("date", "nunique")))

    # Include months even when there are no temperature records
    years = range(int(daily["year"].min()), int(daily["year"].max()) + 1)
    full_calendar = pd.MultiIndex.from_product(
        [years, MONTHS], names=["year", "month"]
    )
    monthly = observed.reindex(full_calendar).reset_index()
    monthly["observed_days"] = monthly["observed_days"].fillna(0).astype(int)
    # Count how many days each month should have
    starts = pd.to_datetime(dict(year=monthly.year, month=monthly.month, day=1))
    monthly["days_in_month"] = starts.dt.days_in_month
    return monthly


monthly_temperatures = summarize_months(daily_temperatures)
ALL_YEARS = sorted(monthly_temperatures["year"].unique().tolist())
# Use the most recent years for the second chart
LEVEL2_YEARS = ALL_YEARS[-LEVEL2_YEAR_COUNT:]
print(f"Level 1: {ALL_YEARS[0]}–{ALL_YEARS[-1]}")
print(f"Level 2: {LEVEL2_YEARS[0]}–{LEVEL2_YEARS[-1]}")

Level 1: 1997–2017
Level 2: 2008–2017


## 4. Shared drawing helpers

Both levels use the same **temperature thresholds**, month labels, layout logic, and hover text. The stepped palette is used for the cell backgrounds and the explanatory legend. Individual daily lines use a shared 0–40 °C y-scale inside every cell.

In [5]:
def make_discrete_colorscale(
    colors: list[str], edges: list[float]
) -> list[list[float | str]]:
    """Build a Plotly colorscale with sharp, non-interpolated band transitions."""
    lower, upper = edges[0], edges[-1]
    scale = []
    for index, color in enumerate(colors):
        start = (edges[index] - lower) / (upper - lower)
        stop = (edges[index + 1] - lower) / (upper - lower)
        # Repeated edges keep the color bands separate
        scale.extend([[start, color], [stop, color]])
    return scale


SPECTRAL_SCALE = make_discrete_colorscale(TEMPERATURE_COLORS, COLOR_BIN_EDGES)


def monthly_grid(monthly: pd.DataFrame, years: list[int], column: str) -> np.ndarray:
    """Return a fixed 12×N matrix indexed by month (row) and year (column)."""
    # Keep the months and years in a consistent order
    return (monthly.pivot(index="month", columns="year", values=column)
            .reindex(index=MONTHS, columns=years).to_numpy(dtype=float))


def monthly_tooltips(monthly: pd.DataFrame, years: list[int]) -> list[list[str]]:
    """Construct month, year, extrema and observation-count hover details."""
    indexed = monthly.set_index(["year", "month"])
    output = []
    for month in MONTHS:
        row = []
        for year in years:
            record = indexed.loc[(year, month)]
            label = f"<b>{calendar.month_name[month]} {year}</b>"
            # Leave missing months without temperature values
            if record["observed_days"] == 0:
                row.append(label + "<br>No recorded temperatures")
            else:
                row.append(
                    f"{label}<br>Monthly maximum: <b>{record['monthly_max']:.0f} °C</b>"
                    f"<br>Monthly minimum: <b>{record['monthly_min']:.0f} °C</b>"
                    f"<br>Days recorded: {int(record['observed_days'])}/"
                    f"{int(record['days_in_month'])}"
                )
        output.append(row)
    return output


def apply_matrix_layout(
    figure: go.Figure, years: list[int], size: tuple[int, int],
    margins: dict[str, int], gap: tuple[int, int]
) -> None:
    """Share the reference's full month labels, top years, and sparse axes."""
    figure.update_layout(
        width=size[0], height=size[1],
        margin=margins,
        paper_bgcolor="white", plot_bgcolor="white",
        font=dict(family="Arial, sans-serif", size=11, color="#111111"),
        hoverlabel=dict(bgcolor="white", font=dict(size=11, color="#161616")),
        template="none",
    )
    # Show the years across the top of the chart
    figure.update_xaxes(
        side="top", tickmode="array", tickvals=years,
        ticktext=[str(year) for year in years],
        tickfont=dict(size=10), ticks="outside", ticklen=5,
        showline=True, linewidth=1, linecolor="black",
        showgrid=False, zeroline=False,
        range=[years[0] - 0.6, years[-1] + 0.6], fixedrange=True,
    )
    # Put January at the top and December at the bottom
    figure.update_yaxes(
        tickmode="array", tickvals=MONTHS, ticktext=MONTH_NAMES,
        tickfont=dict(size=10), ticks="outside", ticklen=5,
        showline=True, linewidth=1, linecolor="black",
        showgrid=False, zeroline=False,
        range=[12.6, 0.4], fixedrange=True,
    )


def add_temperature_legend(figure: go.Figure) -> None:
    """Add a vertical cold-to-hot stepped legend, like level2.png.

    Since the example has 0 at the TOP and 40 at the BOTTOM, we manually
    draw the swatches rather than using Plotly's normally ascending colorbar.
    """
    n = len(TEMPERATURE_COLORS)
    # Draw each band so the legend matches the heatmap
    for index, color in enumerate(TEMPERATURE_COLORS):
        top = 0.99 - index * (0.82 / n)
        bottom = 0.99 - (index + 1) * (0.82 / n)
        figure.add_shape(
            type="rect", xref="paper", yref="paper",
            x0=1.015, x1=1.042, y0=bottom, y1=top,
            fillcolor=color, line=dict(width=0), layer="above",
        )
    figure.add_annotation(xref="paper", yref="paper", x=1.047, y=0.99,
                          text="0 Celsius", showarrow=False,
                          xanchor="left", yanchor="middle", font=dict(size=10))
    figure.add_annotation(xref="paper", yref="paper", x=1.047, y=0.17,
                          text="40 Celsius", showarrow=False,
                          xanchor="left", yanchor="middle", font=dict(size=10))


def temperature_to_y(month: int, temperature: float) -> float:
    """Place a temperature on the shared local y-scale of a month cell.

    The y-axis is reversed so high temperatures move toward the top.
    All months use 0–40 °C; scales are NOT individually rescaled.
    """
    # Keep the same scale for every month with warmer days higher
    t_min, t_max = TEMPERATURE_RANGE
    return month + 0.40 - 0.80 * (temperature - t_min) / (t_max - t_min)

## Level 1 — Year/Month heatmap (1997–2017)

In [6]:
def create_level1_heatmap(monthly: pd.DataFrame) -> go.Figure:
    """Create a reference-style 21-year matrix with a high/low toggle."""
    years = sorted(monthly["year"].unique().tolist())
    tooltips = monthly_tooltips(monthly, years)
    figure = go.Figure()

    # Build one heatmap for highs and another for lows
    for column, name, visible in (
        ("monthly_max", "Monthly maximum", True),
        ("monthly_min", "Monthly minimum", False),
    ):
        figure.add_trace(go.Heatmap(
            x=years, y=MONTHS, z=monthly_grid(monthly, years, column),
            text=tooltips, hovertemplate="%{text}<extra></extra>",
            colorscale=SPECTRAL_SCALE, zmin=0, zmax=40,
            xgap=LEVEL1_GAP[0], ygap=LEVEL1_GAP[1],
            showscale=False, visible=visible, name=name,
            hoverongaps=False,
        ))

    # Match the reference layout and add the color legend
    apply_matrix_layout(
        figure, years, LEVEL1_SIZE,
        margins=dict(l=82, r=100, t=35, b=30), gap=LEVEL1_GAP,
    )
    add_temperature_legend(figure)
    # Switch between the two views without rebuilding the chart
    figure.update_layout(updatemenus=[dict(
        type="buttons", direction="right", active=0, showactive=True,
        x=0.16, y=1.09, xanchor="left", yanchor="bottom",
        bgcolor="white", bordercolor="#cccccc", borderwidth=1,
        font=dict(size=10), pad=dict(t=2, b=2, l=3, r=3),
        buttons=[
            dict(label="Monthly maximum", method="update",
                 args=[{"visible": [True, False]}]),
            dict(label="Monthly minimum", method="update",
                 args=[{"visible": [False, True]}]),
        ]
    )])
    return figure


level1_figure = create_level1_heatmap(monthly_temperatures)
level1_figure.show()

## Level 2 — Year/Month heatmap with daily mini line charts (2008–2017)

In [7]:
def build_daily_line_data(
    daily: pd.DataFrame, years: list[int]
) -> dict[str, dict[str, list]]:
    """Create efficient high/low line traces with gaps for missing dates.

    Using just two combined Plotly traces keeps this relatively large chart
    responsive. None separators prevent connecting unrelated months/years.
    """
    rows = daily[daily["year"].isin(years)].set_index("date")
    line_data = {
        "max_temperature": {"x": [], "y": [], "date": [], "temperature": []},
        "min_temperature": {"x": [], "y": [], "date": [], "temperature": []},
    }

    # Go through each month and put its days inside the right cell
    for year in years:
        for month in MONTHS:
            month_length = calendar.monthrange(year, month)[1]
            for day in range(1, month_length + 1):
                date = pd.Timestamp(year=year, month=month, day=day)
                # Spread the daily readings across the cell
                x = year - 0.39 + (day - 1) * (0.78 / 30)
                for metric in line_data:
                    series = line_data[metric]
                    series["x"].append(x)
                    series["date"].append(date.strftime("%Y-%m-%d"))
                    value = rows.at[date, metric] if date in rows.index else None
                    series["temperature"].append(value)
                    series["y"].append(
                        temperature_to_y(month, value) if value is not None else None
                    )

            # Separate the lines between months
            for series in line_data.values():
                series["x"].append(None)
                series["y"].append(None)
                series["date"].append(None)
                series["temperature"].append(None)
    return line_data


def create_level2_daily_matrix(
    daily: pd.DataFrame, monthly: pd.DataFrame, year_count: int = 10
) -> go.Figure:
    """Draw the ten-year monthly heatmap with green/blue daily mini charts."""
    all_years = sorted(daily["year"].unique().tolist())
    years = all_years[-year_count:]
    selected_months = monthly[monthly["year"].isin(years)]

    figure = go.Figure()
    # Use the monthly maximum for the background
    figure.add_trace(go.Heatmap(
        x=years, y=MONTHS,
        z=monthly_grid(selected_months, years, "monthly_max"),
        text=monthly_tooltips(selected_months, years),
        hovertemplate="%{text}<extra></extra>",
        colorscale=SPECTRAL_SCALE, zmin=0, zmax=40,
        xgap=LEVEL2_GAP[0], ygap=LEVEL2_GAP[1],
        showscale=False, hoverongaps=False,
        name="Monthly maximum (background)",
    ))

    # Use the same temperature scale for every month
    lines = build_daily_line_data(daily, years)
    for metric, label, color in (
        ("max_temperature", "Daily maximum", DAILY_MAX_COLOR),
        ("min_temperature", "Daily minimum", DAILY_MIN_COLOR),
    ):
        points = lines[metric]
        figure.add_trace(go.Scatter(
            x=points["x"], y=points["y"],
            mode="lines", line=dict(color=color, width=1.6),
            customdata=list(zip(points["date"], points["temperature"])),
            hovertemplate=(f"%{{customdata[0]}}<br>{label}: "
                           f"<b>%{{customdata[1]}} °C</b><extra></extra>"),
            showlegend=True, name=label, connectgaps=False,
            legendgroup=metric,
        ))

    # Finish the chart with the shared axes and temperature legend
    apply_matrix_layout(
        figure, years, LEVEL2_SIZE,
        margins=dict(l=78, r=105, t=35, b=35), gap=LEVEL2_GAP,
    )
    add_temperature_legend(figure)
    # Place the daily line labels below the plot
    figure.update_layout(
        legend=dict(orientation="h", yanchor="top", y=-0.022,
                    xanchor="center", x=0.50, font=dict(size=11),
                    bgcolor="rgba(255,255,255,0)"),
        hovermode="closest",
    )
    return figure


level2_figure = create_level2_daily_matrix(
    daily_temperatures, monthly_temperatures, LEVEL2_YEAR_COUNT
)
level2_figure.show()

## 5. Correctness checks

In [8]:
def validate_visualizations(
    daily: pd.DataFrame, monthly: pd.DataFrame,
    level1: go.Figure, level2: go.Figure
) -> None:
    """Assert key requirements and data invariants for both challenges."""
    # Check that all years and months are present
    assert len(monthly) == len(ALL_YEARS) * 12
    assert len(ALL_YEARS) == 21 and ALL_YEARS[0] == 1997
    assert LEVEL2_YEARS == list(range(2008, 2018))

    # Compare one month with known values from the data
    march_2002 = monthly.query("year == 2002 and month == 3").iloc[0]
    assert march_2002["monthly_max"] == 28
    assert march_2002["monthly_min"] == 14

    # Check the two heatmaps and the view buttons
    assert len(level1.data) == 2
    assert np.asarray(level1.data[0].z).shape == (12, 21)
    assert np.asarray(level1.data[1].z).shape == (12, 21)
    assert len(level1.layout.updatemenus[0].buttons) == 2

    # Check the background and both daily line traces
    assert len(level2.data) == 3
    assert np.asarray(level2.data[0].z).shape == (12, 10)
    assert level2.data[1].line.color == DAILY_MAX_COLOR
    assert level2.data[2].line.color == DAILY_MIN_COLOR
    assert level2.data[1].name == "Daily maximum"
    assert level2.data[2].name == "Daily minimum"

    # Count each day plus a gap after every month
    expected_points = sum(calendar.monthrange(year, month)[1]
                          for year in LEVEL2_YEARS for month in MONTHS) + 120
    assert len(level2.data[1].x) == expected_points
    assert len(level2.data[2].x) == expected_points

    # Keep months without data empty
    assert np.isnan(level2.data[0].z[10][9])
    assert np.isnan(level2.data[0].z[11][9])
    assert daily['date'].max() == pd.Timestamp('2017-10-28')
    print("All checks passed: correct monthly extrema, years, heatmap grids, "
          "daily traces, legends, and missing months.")


validate_visualizations(
    daily_temperatures, monthly_temperatures, level1_figure, level2_figure
)

All checks passed: correct monthly extrema, years, heatmap grids, daily traces, legends, and missing months.
